# DPO 訓練（Colab 版）

跑之前先確認：Runtime → Change runtime type → Hardware accelerator 選 GPU（Pro 帳號的話 GPU class 選 **Standard** 即可，通常會配到 16GB VRAM 的 T4，不需要 Premium 的 A100）。

第一次設定：把本機 `dpo/` 資料夾裡的 `.py` 腳本、`scenarios.json`、`requirements.txt`，加上 `data/train.jsonl`，上傳到你自己 Google Drive 的一個資料夾（例如 `我的雲端硬碟/DPO/`）。**`train.jsonl` 一定要放在這個資料夾底下的 `data/` 子資料夾裡**（`train_dpo.py` 裡寫死要讀 `Path(__file__).parent / "data" / "train.jsonl"`），不能直接跟 `.py` 檔案放同一層。`models/`（15GB 本地模型）跟 `output/`（訓練產物）不用上傳，下面的 cell 會自動處理。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 改成你在 Drive 上實際存放這些腳本/資料的資料夾路徑（.py 檔案直接在這一層底下，
# 不是再包一層 dpo/ 子資料夾）
PROJECT_DIR = '/content/drive/MyDrive/DPO'

In [ ]:
# 確認拿到的 GPU 與 VRAM
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 安裝訓練需要的套件（Colab 預設沒有這些，dpo/requirements.txt 裡 GPU 那幾行是註解掉的）
!pip install -q unsloth trl "transformers>=4.40.0" "peft>=0.10.0" "bitsandbytes>=0.43.0" "datasets>=2.0.0"

## 模型（只需要做一次）

本地模型 15GB，直接下載進 Drive 掛載路徑會被 Drive 寫入速度拖慢，所以先下載到 Colab 本機硬碟 `/content/`，下載完再搬進 Drive。之後每次開新 session，`models/taiwan-llama` 已經在 Drive 裡了，跳過這一段、直接跑下面的訓練 cell 即可。

In [ ]:
import os

MODEL_DIR = f'{PROJECT_DIR}/models/taiwan-llama'

if os.path.exists(MODEL_DIR):
    print(f'模型已存在於 {MODEL_DIR}，跳過下載。')
else:
    print('模型不存在，先下載到本機硬碟再搬進 Drive（第一次執行會花一些時間）...')
    !python {PROJECT_DIR}/download_model.py --local-dir /content/dpo_model_tmp
    !mkdir -p {PROJECT_DIR}/models
    !cp -r /content/dpo_model_tmp {MODEL_DIR}
    print('模型已搬進 Drive，之後 session 不用再下載。')

## 訓練前資料驗證（可選，train_dpo.py 內建會自動跑一次）

In [ ]:
%cd {PROJECT_DIR}
!python validate_data.py

## 開始訓練

checkpoint 跟 LoRA adapter 會存進 `PROJECT_DIR/output/`，因為是相對 `__file__` 算的路徑，本來就在 Drive 掛載範圍內——就算 session 斷線，`save_strategy="epoch"` 存的 checkpoint 也不會丟。

In [ ]:
%cd {PROJECT_DIR}
!python train_dpo.py